In [4]:
import pandas as pd
import numpy as np
from typing import Optional

PARAM_CSV_HOMO = (
    "/opt/domain-adaptation-people-counting/results/"
    "parameter_based_transfer/all_parameter_based_homogeneous.csv"
)

PARAM_CSV_HETERO = (
    "/opt/domain-adaptation-people-counting/results/"
    "parameter_based_transfer/all_parameter_based_heterogeneous.csv"
)

# ------------------------------------------------------------
# Map result-file method names to paper-table method names
# Keep TSW and TSBW as separate columns
# Keep only LinInt(alpha=0.50) and LinInt(alpha=0.75)
# ------------------------------------------------------------
def map_method_for_paper(method_name: str) -> Optional[str]:
    # Uses FT? = Yes
    if method_name == "Full-Fine-Tuning":
        return "Full-Fine-Tuning"
    if method_name == "Feature-Extraction":
        return "Feature-Extraction"
    if method_name == "Label-Aware Joint FT (tsw)":
        return "LA-Joint FT (TSW)"
    if method_name == "Label-Aware Joint FT (tsbw)":
        return "LA-Joint FT (TSBW)"
    if method_name == "Label-Aware Pretrain to Target FT (tsw)":
        return "LA-Pretrain → TgtFT (TSW)"
    if method_name == "Label-Aware Pretrain to Target FT (tsbw)":
        return "LA-Pretrain → TgtFT (TSBW)"
    if method_name == "Target-Priority Joint FT (tsw)":
        return "Target-Priority Joint FT (TSW)"
    if method_name == "Target-Priority Joint FT (tsbw)":
        return "Target-Priority Joint FT (TSBW)"
    if method_name == "Thresholded Label-Aware FT (tsw)":
        return "Thresh-LA FT (TSW)"
    if method_name == "Thresholded Label-Aware FT (tsbw)":
        return "Thresh-LA FT (TSBW)"
    # Uses FT? = No
    if method_name == "Direct-Transfer":
        return "Direct Transfer"
    if method_name == "Joint-Training":
        return "Joint-Training"
    if method_name == "Label-Aware Source Training (tsw)":
        return "LA-SrcTraining (TSW)"
    if method_name == "Label-Aware Source Training (tsbw)":
        return "LA-SrcTraining (TSBW)"
    if method_name == "Thresholded Label-Aware Source Training (tsw)":
        return "Thresh-LA-SrcTraining (TSW)"
    if method_name == "Thresholded Label-Aware Source Training (tsbw)":
        return "Thresh-LA-SrcTraining (TSBW)"
    if method_name == "LinInt (alpha=0.50)":
        return "LinInt (0.50)"
    if method_name == "LinInt (alpha=0.75)":
        return "LinInt (0.75)"
    if method_name == "Source Model + Target Calibration":
        return "SrcModel + TgtCalibration"
    if method_name == "Source-to-Target Linear Mapping":
        return "Source-to-TgtLinear Mapping"
    if method_name == "Target-Only":
        return "Target-Only"
    return None


# ------------------------------------------------------------
# Column order matching your method table order
# ------------------------------------------------------------
method_order = [
    # Uses FT? = Yes
    "Full-Fine-Tuning",
    "Feature-Extraction",
    "LA-Joint FT (TSW)",
    "LA-Joint FT (TSBW)",
    "LA-Pretrain → TgtFT (TSW)",
    "LA-Pretrain → TgtFT (TSBW)",
    "Target-Priority Joint FT (TSW)",
    "Target-Priority Joint FT (TSBW)",
    "Thresh-LA FT (TSW)",
    "Thresh-LA FT (TSBW)",
    # Uses FT? = No
    "Direct Transfer",
    "Joint-Training",
    "LA-SrcTraining (TSW)",
    "LA-SrcTraining (TSBW)",
    "Thresh-LA-SrcTraining (TSW)",
    "Thresh-LA-SrcTraining (TSBW)",
    "LinInt (0.50)",
    "LinInt (0.75)",
    "SrcModel + TgtCalibration",
    "Source-to-TgtLinear Mapping",
    "Target-Only",
]

backbone_order = ["RNN", "LSTM", "Transformer"]


# ------------------------------------------------------------
# Shared helper — builds display_df from a CSV path
# ------------------------------------------------------------
def build_table(param_csv: str):
    df = pd.read_csv(param_csv)

    df["paper_method"] = df["method"].apply(map_method_for_paper)
    paper_df = df[df["paper_method"].notna()].copy()

    agg_df = (
        paper_df.groupby(["scenario", "model", "paper_method"], as_index=False)["test_mae"]
        .min()
    )

    scenario_order = list(dict.fromkeys(agg_df["scenario"].tolist()))

    wide = agg_df.pivot_table(
        index=["scenario", "model"],
        columns="paper_method",
        values="test_mae",
        aggfunc="first"
    ).reset_index()

    for col in method_order:
        if col not in wide.columns:
            wide[col] = np.nan

    wide = wide[["scenario", "model"] + method_order]

    wide["scenario"] = pd.Categorical(wide["scenario"], categories=scenario_order, ordered=True)
    wide["model"]    = pd.Categorical(wide["model"],    categories=backbone_order, ordered=True)
    wide = wide.sort_values(["scenario", "model"]).reset_index(drop=True)

    wide = wide.rename(columns={"scenario": "Setting", "model": "Backbone"})

    display_df = wide.copy()
    for col in method_order:
        display_df[col] = display_df[col].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")

    return display_df


# ============================================================
# TABLE 1 — HOMOGENEOUS
# ============================================================
display_homo = build_table(PARAM_CSV_HOMO)

print("\nPaper-style MAE table — HOMOGENEOUS — separate TSW / TSBW columns:\n")
print(display_homo.to_string(index=False))

display_homo.to_csv("parameter_based_paper_table_homogeneous.csv", index=False)

latex_str = display_homo.to_latex(index=False, escape=False)
with open("parameter_based_paper_table_homogeneous.tex", "w") as f:
    f.write(latex_str)

print("\nSaved:")
print(" - parameter_based_paper_table_homogeneous.csv")
print(" - parameter_based_paper_table_homogeneous.tex")


# ============================================================
# TABLE 2 — HETEROGENEOUS
# ============================================================
display_hetero = build_table(PARAM_CSV_HETERO)

print("\nPaper-style MAE table — HETEROGENEOUS — separate TSW / TSBW columns:\n")
print(display_hetero.to_string(index=False))

display_hetero.to_csv("parameter_based_paper_table_heterogeneous.csv", index=False)

latex_str = display_hetero.to_latex(index=False, escape=False)
with open("parameter_based_paper_table_heterogeneous.tex", "w") as f:
    f.write(latex_str)

print("\nSaved:")
print(" - parameter_based_paper_table_heterogeneous.csv")
print(" - parameter_based_paper_table_heterogeneous.tex")


Paper-style MAE table — HOMOGENEOUS — separate TSW / TSBW columns:

         Setting    Backbone Full-Fine-Tuning Feature-Extraction LA-Joint FT (TSW) LA-Joint FT (TSBW) LA-Pretrain → TgtFT (TSW) LA-Pretrain → TgtFT (TSBW) Target-Priority Joint FT (TSW) Target-Priority Joint FT (TSBW) Thresh-LA FT (TSW) Thresh-LA FT (TSBW) Direct Transfer Joint-Training LA-SrcTraining (TSW) LA-SrcTraining (TSBW) Thresh-LA-SrcTraining (TSW) Thresh-LA-SrcTraining (TSBW) LinInt (0.50) LinInt (0.75) SrcModel + TgtCalibration Source-to-TgtLinear Mapping Target-Only
Room_A -> Room_B         RNN            0.261              0.586             0.371              0.329                     0.253                      0.234                          0.347                           0.297              0.436               0.290           0.700          0.504                0.585                 0.521                       0.564                        0.396         0.391         0.255                     0.513        

In [17]:
"""
Parameter-Based Transfer — Grouped Bar Chart (Publication Quality)

3 figures: RNN | LSTM | Transformer

X-axis layout:
  [A→B: 10 FT bars | small gap | 11 Non-FT bars]  BIG GAP
  [A→P: 10 FT bars | small gap | 11 Non-FT bars]  BIG GAP
  ...

- Narrow bars, tight spacing — fits A4 / double-column paper
- Exact MAE on every bar (rotated, small font)
- ★ + black border on best method per setting
- Two separate legends (FT / Non-FT)
- Grey hatched bar for N/A methods

Author: Azadshokrollahi
"""

import sys
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator
from pathlib import Path
from typing import Optional

matplotlib.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "DejaVu Serif"],
    "font.size":          8,
    "axes.titlesize":     9,
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    7,
    "legend.title_fontsize": 7.5,
    "figure.dpi":         150,
    "savefig.dpi":        600,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.width":  0.8,
    "ytick.major.width":  0.8,
})

# ─── sys.path fix ─────────────────────────────────────────────────────────
try:
    _SRC_DIR = Path(__file__).resolve().parent.parent / "src"
except NameError:
    _SRC_DIR = next(
        (p / "src"
         for p in [Path.cwd(), Path.cwd().parent]
         if (p / "src" / "config.py").exists()),
        Path.cwd(),
    )
if str(_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(_SRC_DIR))

from config import RESULTS_ROOT

# ─── Paths ────────────────────────────────────────────────────────────────
PARAM_DIR  = RESULTS_ROOT / "parameter_based_transfer"
OUTPUT_DIR = PARAM_DIR / "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PARAM_CSV_HOMO   = PARAM_DIR / "all_parameter_based_homogeneous.csv"
PARAM_CSV_HETERO = PARAM_DIR / "all_parameter_based_heterogeneous.csv"

##############################################################################
#  METHOD MAP
##############################################################################

def map_method(name: str) -> Optional[str]:
    _MAP = {
        "Full-Fine-Tuning":                               "Full-FT",
        "Feature-Extraction":                             "Feat-Ext",
        "Label-Aware Joint FT (tsw)":                     "LA-JFT (TSW)",
        "Label-Aware Joint FT (tsbw)":                    "LA-JFT (TSBW)",
        "Label-Aware Pretrain to Target FT (tsw)":        "LA-Pre→FT (TSW)",
        "Label-Aware Pretrain to Target FT (tsbw)":       "LA-Pre→FT (TSBW)",
        "Target-Priority Joint FT (tsw)":                 "TgtPrio (TSW)",
        "Target-Priority Joint FT (tsbw)":                "TgtPrio (TSBW)",
        "Thresholded Label-Aware FT (tsw)":               "Thr-LA-FT (TSW)",
        "Thresholded Label-Aware FT (tsbw)":              "Thr-LA-FT (TSBW)",
        "Direct-Transfer":                                "Direct-Tr.",
        "Joint-Training":                                 "Joint-Tr.",
        "Label-Aware Source Training (tsw)":              "LA-Src (TSW)",
        "Label-Aware Source Training (tsbw)":             "LA-Src (TSBW)",
        "Thresholded Label-Aware Source Training (tsw)":  "Thr-Src (TSW)",
        "Thresholded Label-Aware Source Training (tsbw)": "Thr-Src (TSBW)",
        "LinInt (alpha=0.50)":                            "LinInt (0.50)",
        "LinInt (alpha=0.75)":                            "LinInt (0.75)",
        "Source Model + Target Calibration":              "Src+Calib",
        "Source-to-Target Linear Mapping":                "Src→LinMap",
        "Target-Only":                                    "Tgt-Only",
    }
    return _MAP.get(name, None)


##############################################################################
#  GROUPS, SCENARIOS, COLORS
##############################################################################

FT_METHODS = [
    "Full-FT",
    "Feat-Ext",
    "LA-JFT (TSW)",
    "LA-JFT (TSBW)",
    "LA-Pre→FT (TSW)",
    "LA-Pre→FT (TSBW)",
    "TgtPrio (TSW)",
    "TgtPrio (TSBW)",
    "Thr-LA-FT (TSW)",
    "Thr-LA-FT (TSBW)",
]

NON_FT_METHODS = [
    "Direct-Tr.",
    "Joint-Tr.",
    "LA-Src (TSW)",
    "LA-Src (TSBW)",
    "Thr-Src (TSW)",
    "Thr-Src (TSBW)",
    "LinInt (0.50)",
    "LinInt (0.75)",
    "Src+Calib",
    "Src→LinMap",
    "Tgt-Only",
]

ALL_METHODS = FT_METHODS + NON_FT_METHODS
N_FT        = len(FT_METHODS)      # 10
N_NON_FT    = len(NON_FT_METHODS)  # 11

SCENARIO_ORDER = [
    "Room_A -> Room_B",
    "Room_A -> Room_P",
    "Room_B -> Room_C",
    "Room_B -> Room_P",
    "Room_C -> Room_A",
    "Room_C -> Room_P",
    "Room_P -> Room_A",
    "Room_P -> Room_B",
    "Room_P -> Room_C",
]

SCENARIO_SHORT = {
    "Room_A -> Room_B": "A→B",
    "Room_A -> Room_P": "A→P",
    "Room_B -> Room_C": "B→C",
    "Room_B -> Room_P": "B→P",
    "Room_C -> Room_A": "C→A",
    "Room_C -> Room_P": "C→P",
    "Room_P -> Room_A": "P→A",
    "Room_P -> Room_B": "P→B",
    "Room_P -> Room_C": "P→C",
}

BACKBONE_ORDER = ["RNN", "LSTM", "Transformer"]

# ── Colors — colorblind-friendly palette ──────────────────────────────────
# 10 for FT, 11 for Non-FT
FT_COLORS = [
    "#D62728",  # red
    "#FF7F0E",  # orange
    "#FFBB78",  # light orange
    "#2CA02C",  # green
    "#98DF8A",  # light green
    "#1F77B4",  # blue
    "#AEC7E8",  # light blue
    "#9467BD",  # purple
    "#C5B0D5",  # light purple
    "#E377C2",  # pink
]

NON_FT_COLORS = [
    "#8C564B",  # brown
    "#C49C94",  # light brown
    "#17BECF",  # cyan
    "#9EDAE5",  # light cyan
    "#7F7F7F",  # grey
    "#BCBD22",  # yellow-green
    "#DBDB8D",  # light yellow-green
    "#F7B6D2",  # light pink
    "#FF9896",  # salmon
    "#C7C7C7",  # light grey
    "#393B79",  # dark blue
]

# ── Spacing (narrow — paper-ready) ────────────────────────────────────────
BAR_W       = 0.18   # bar width
INNER_GAP   = 0.10   # gap between FT and Non-FT within same setting
SETTING_GAP = 0.60   # big gap between consecutive settings


##############################################################################
#  COMPUTE X-POSITIONS
##############################################################################

def compute_x_positions():
    ft_x, nft_x       = [], []
    label_x            = []
    divider_x          = []
    setting_div        = []

    cursor = 0.0
    for s_idx in range(len(SCENARIO_ORDER)):
        # FT bar positions
        s_ft = [cursor + m * BAR_W for m in range(N_FT)]
        ft_end = s_ft[-1]

        # Non-FT bar positions (after inner gap)
        nft_start = ft_end + BAR_W + INNER_GAP
        s_nft = [nft_start + m * BAR_W for m in range(N_NON_FT)]
        nft_end = s_nft[-1]

        ft_x.append(s_ft)
        nft_x.append(s_nft)

        # Center label = midpoint of entire group
        label_x.append((s_ft[0] + s_nft[-1]) / 2)

        # Inner divider (dashed) between FT and Non-FT
        divider_x.append((ft_end + nft_start) / 2)

        # Big solid divider between settings
        if s_idx < len(SCENARIO_ORDER) - 1:
            setting_div.append(nft_end + BAR_W / 2 + SETTING_GAP / 2)

        cursor = nft_end + BAR_W + SETTING_GAP

    return ft_x, nft_x, label_x, divider_x, setting_div


##############################################################################
#  LOAD + PIVOT
##############################################################################

def load_pivot(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["paper_method"] = df["method"].apply(map_method)
    df = df[df["paper_method"].notna()].copy()

    agg = (
        df.groupby(["scenario", "model", "paper_method"], as_index=False)["test_mae"]
        .min()
    )

    wide = agg.pivot_table(
        index=["scenario", "model"],
        columns="paper_method",
        values="test_mae",
        aggfunc="first",
    ).reset_index()

    for col in ALL_METHODS:
        if col not in wide.columns:
            wide[col] = np.nan

    return wide[["scenario", "model"] + ALL_METHODS]


##############################################################################
#  DRAW BARS (shared for FT and Non-FT blocks)
##############################################################################

def draw_bars(ax, x_positions, methods, colors, row,
              best_val, ymax, hatch_na=True):
    """Draw one group of bars (either FT or Non-FT) for one scenario row."""
    for m_idx, (method, color, x) in enumerate(
            zip(methods, colors, x_positions)):
        val = row[method] if method in row.index else np.nan

        # ── N/A bar ───────────────────────────────────────────────────
        if np.isnan(val):
            if hatch_na:
                ax.bar(x, ymax * 0.04,
                       width=BAR_W,
                       color="#F0F0F0",
                       edgecolor="#BBBBBB",
                       linewidth=0.5,
                       hatch="///",
                       zorder=2)
                ax.text(x, ymax * 0.05, "n/a",
                        ha="center", va="bottom",
                        fontsize=3.5, color="#AAAAAA",
                        rotation=90)
            continue

        is_best = abs(val - best_val) < 1e-9

        # ── Bar ───────────────────────────────────────────────────────
        ax.bar(x, val,
               width=BAR_W,
               color=color,
               edgecolor="black" if is_best else "none",
               linewidth=1.5  if is_best else 0.0,
               alpha=0.90,
               zorder=3)

        # ── Exact value label ─────────────────────────────────────────
        label = f"★{val:.3f}" if is_best else f"{val:.3f}"
        ax.text(x, val + ymax * 0.006,
                label,
                ha="center", va="bottom",
                fontsize=3.8,
                fontweight="bold" if is_best else "normal",
                color="black" if is_best else color,
                rotation=90,
                zorder=5)


##############################################################################
#  MAIN PLOT — one figure per backbone
##############################################################################

def plot_grouped_bar(wide_df: pd.DataFrame,
                      backbone: str,
                      variant: str):
    sub = wide_df[wide_df["model"] == backbone].copy()
    if len(sub) == 0:
        print(f"  No data for {backbone} / {variant}")
        return

    ft_x, nft_x, label_x, divider_x, setting_div = compute_x_positions()

    # ── Global y range ────────────────────────────────────────────────────
    all_vals = sub[ALL_METHODS].values.astype(float)
    ymax     = round(float(np.nanmax(all_vals)) * 1.30, 1)
    ymin     = 0.0

    # ── Figure size — narrow, tall enough for labels ──────────────────────
    n_total_bars = len(SCENARIO_ORDER) * (N_FT + N_NON_FT)
    fig_w = max(22, n_total_bars * BAR_W * 1.15)
    fig_h = 8.0   # standard journal column height

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.subplots_adjust(
        bottom=0.28, top=0.91,
        left=0.05,  right=0.99,
    )

    # ── Draw bars for every scenario ─────────────────────────────────────
    for s_idx, scenario in enumerate(SCENARIO_ORDER):
        rows = sub[sub["scenario"] == scenario]
        if len(rows) == 0:
            continue
        row = rows.iloc[0]

        best_val = float(np.nanmin(
            [row[m] for m in ALL_METHODS if m in row.index]))

        # FT group
        draw_bars(ax, ft_x[s_idx],  FT_METHODS,     FT_COLORS,
                  row, best_val, ymax)

        # Non-FT group
        draw_bars(ax, nft_x[s_idx], NON_FT_METHODS, NON_FT_COLORS,
                  row, best_val, ymax)

        # ── Inner dashed divider (FT | Non-FT) ───────────────────────
        ax.axvline(divider_x[s_idx],
                   color="#888888", linewidth=0.8,
                   linestyle="--", alpha=0.7, zorder=4)

        # ── Sub-group labels beneath x-axis ──────────────────────────
        ft_cx  = (ft_x[s_idx][0]  + ft_x[s_idx][-1])  / 2
        nft_cx = (nft_x[s_idx][0] + nft_x[s_idx][-1]) / 2

        ax.annotate("FT-Based",
                    xy=(ft_cx, 0), xycoords=("data", "axes fraction"),
                    xytext=(0, -22), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=5.5, color="#1A5276",
                    fontweight="bold",
                    annotation_clip=False)

        ax.annotate("Non-FT",
                    xy=(nft_cx, 0), xycoords=("data", "axes fraction"),
                    xytext=(0, -22), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=5.5, color="#784212",
                    fontweight="bold",
                    annotation_clip=False)

    # ── Big solid setting dividers ─────────────────────────────────────
    for sdx in setting_div:
        ax.axvline(sdx, color="black",
                   linewidth=1.6, linestyle="-",
                   alpha=0.9, zorder=6)

    # ── X-axis: scenario labels ───────────────────────────────────────
    ax.set_xticks(label_x)
    ax.set_xticklabels(
        [SCENARIO_SHORT[s] for s in SCENARIO_ORDER],
        fontsize=9, fontweight="bold",
    )
    ax.tick_params(axis="x", length=0, pad=28)

    # ── Y-axis ───────────────────────────────────────────────────────
    ax.set_ylabel("Test MAE", fontsize=9, fontweight="bold")
    ax.set_ylim(ymin, ymax)
    ax.yaxis.set_minor_locator(MultipleLocator(0.05))
    ax.grid(True, axis="y", which="major",
            alpha=0.35, linestyle="--", linewidth=0.7, zorder=0)
    ax.grid(True, axis="y", which="minor",
            alpha=0.15, linestyle=":", linewidth=0.5, zorder=0)

    ax.set_xlim(ft_x[0][0] - BAR_W * 0.8,
                nft_x[-1][-1] + BAR_W * 0.8)

    # ── Title ────────────────────────────────────────────────────────
    fig.suptitle(
        f"Parameter-Based Transfer — Test MAE  |  "
        f"Backbone: {backbone}  |  {variant.capitalize()}  |  ★ best per setting",
        fontsize=10, fontweight="bold", y=0.97,
    )

    # ── Legends ──────────────────────────────────────────────────────
    ft_handles = [
        mpatches.Patch(facecolor=FT_COLORS[i],
                       edgecolor="none", label=m, alpha=0.90)
        for i, m in enumerate(FT_METHODS)
    ]
    nft_handles = [
        mpatches.Patch(facecolor=NON_FT_COLORS[i],
                       edgecolor="none", label=m, alpha=0.90)
        for i, m in enumerate(NON_FT_METHODS)
    ]
    na_handle = mpatches.Patch(
        facecolor="#F0F0F0", edgecolor="#BBBBBB",
        hatch="///", label="N/A (not applicable)")

    leg_ft = ax.legend(
        handles=ft_handles,
        title="Fine-Tuning-Based",
        loc="upper left",
        bbox_to_anchor=(0.0, -0.14),
        ncol=5,
        fontsize=6.8,
        frameon=True,
        edgecolor="#2E86C1",
        framealpha=0.95,
        bbox_transform=ax.transAxes,
    )
    ax.add_artist(leg_ft)

    ax.legend(
        handles=nft_handles + [na_handle],
        title="Non-Fine-Tuning",
        loc="upper right",
        bbox_to_anchor=(1.0, -0.14),
        ncol=4,
        fontsize=6.8,
        frameon=True,
        edgecolor="#E67E22",
        framealpha=0.95,
        bbox_transform=ax.transAxes,
    )

    # ── Save ─────────────────────────────────────────────────────────
    fname = f"grouped_bar_{variant}_{backbone}"
    for ext in ["pdf", "png"]:
        out = OUTPUT_DIR / f"{fname}.{ext}"
        fig.savefig(out, dpi=600, bbox_inches="tight")
        print(f"    Saved: {out}")

    plt.show()
    plt.close()


##############################################################################
#  MAIN
##############################################################################

def main():
    print("=" * 70)
    print("GROUPED BAR — Publication Quality")
    print(f"  Output: {OUTPUT_DIR}")
    print("=" * 70)

    for variant, csv_path in [
        ("homogeneous",   PARAM_CSV_HOMO),
        ("heterogeneous", PARAM_CSV_HETERO),
    ]:
        if not csv_path.exists():
            print(f"  ⚠  {csv_path} not found — skipped.")
            continue

        print(f"\n══ {variant.upper()} ══")
        wide = load_pivot(csv_path)

        for backbone in BACKBONE_ORDER:
            print(f"\n  Backbone: {backbone}")
            plot_grouped_bar(wide, backbone, variant)

    print("\n" + "=" * 70)
    print("✓ DONE →", OUTPUT_DIR)
    print("=" * 70)


if __name__ == "__main__":
    main()

GROUPED BAR — Publication Quality
  Output: /opt/domain-adaptation-people-counting/results/parameter_based_transfer/figures
  ⚠  /opt/domain-adaptation-people-counting/results/parameter_based_transfer/all_parameter_based_homogeneous.csv not found — skipped.
  ⚠  /opt/domain-adaptation-people-counting/results/parameter_based_transfer/all_parameter_based_heterogeneous.csv not found — skipped.

✓ DONE → /opt/domain-adaptation-people-counting/results/parameter_based_transfer/figures


In [ ]:
"""
Parameter-Based Transfer — Critical Difference (CD) Diagram
Friedman omnibus test + Nemenyi post-hoc  (Demsar, JMLR 2006).

Why this instead of a grouped bar chart?
  Comparing 21 methods over many settings with raw bars is unreadable and
  says nothing about significance. A CD diagram is the standard, peer-reviewed
  way to argue "method X is (significantly) better than Y across datasets".

Protocol
  • Each (Setting x Backbone) pair is treated as ONE independent "dataset".
  • Within each dataset the methods are ranked by Test MAE
    (rank 1 = lowest MAE = best). Ties get the average rank.
  • The Friedman test checks whether the methods differ at all.
  • Nemenyi post-hoc gives a single Critical Difference (CD): two methods are
    statistically indistinguishable at level alpha if their AVERAGE ranks
    differ by less than CD. Such methods are joined by a bold horizontal bar.

Reads (produced by the first cell of this notebook):
  • parameter_based_paper_table_homogeneous.csv
  • parameter_based_paper_table_heterogeneous.csv

Saves (PDF + PNG, 600 dpi):
  • results/parameter_based_transfer/figures/cd_diagram_homogeneous.{pdf,png}
  • results/parameter_based_transfer/figures/cd_diagram_heterogeneous.{pdf,png}

Author: Azadshokrollahi
"""

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare

# ── Publication style ───────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":        9,
    "axes.titlesize":   10,
    "figure.dpi":       150,
    "savefig.dpi":      600,
})

ALPHA      = 0.05                       # significance level
META_COLS  = ["Setting", "Backbone"]
TABLES     = {
    "homogeneous":   "parameter_based_paper_table_homogeneous.csv",
    "heterogeneous": "parameter_based_paper_table_heterogeneous.csv",
}
OUT_DIR = Path("..") / "results" / "parameter_based_transfer" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
#  Nemenyi critical value q_alpha  (already divided by sqrt(2))
#  Primary: scipy studentized_range.  Fallback: tabulated values.
# ─────────────────────────────────────────────────────────────────────────────
_Q_TABLE = {
    0.05: {2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031,9:3.102,
           10:3.164,11:3.219,12:3.268,13:3.313,14:3.354,15:3.391,16:3.426,
           17:3.458,18:3.489,19:3.517,20:3.544,21:3.569,22:3.593,23:3.616,
           24:3.637,25:3.658,26:3.678,27:3.696,28:3.714,29:3.732,30:3.749},
    0.10: {2:1.645,3:2.052,4:2.291,5:2.460,6:2.589,7:2.693,8:2.780,9:2.855,
           10:2.920,11:2.978,12:3.030,13:3.077,14:3.120,15:3.159,16:3.196,
           17:3.230,18:3.261,19:3.291,20:3.319,21:3.346,22:3.371,23:3.394,
           24:3.417,25:3.439,26:3.459,27:3.479,28:3.498,29:3.516,30:3.533},
}


def q_alpha(k: int, alpha: float) -> float:
    """Critical value for the Nemenyi test (Studentized range / sqrt(2))."""
    try:
        from scipy.stats import studentized_range
        return float(studentized_range.ppf(1.0 - alpha, k, np.inf)) / math.sqrt(2.0)
    except Exception:
        return _Q_TABLE[alpha][k]


# ─────────────────────────────────────────────────────────────────────────────
#  Load aggregated MAE matrix  (rows = datasets, cols = methods)
# ─────────────────────────────────────────────────────────────────────────────
def load_matrix(path: str):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found — run the first cell of this notebook to (re)build "
            f"the paper tables, then re-run this cell."
        )
    df = pd.read_csv(p)
    method_cols = [c for c in df.columns if c not in META_COLS]
    mae = df[method_cols].apply(pd.to_numeric, errors="coerce")

    # CD requires a complete matrix — keep only methods defined for EVERY dataset
    keep    = [c for c in method_cols if mae[c].notna().all()]
    dropped = [c for c in method_cols if c not in keep]
    return mae[keep], keep, dropped, len(df)


# ─────────────────────────────────────────────────────────────────────────────
#  Cliques = maximal groups of methods that are NOT significantly different
# ─────────────────────────────────────────────────────────────────────────────
def find_cliques(ranks_sorted: np.ndarray, cd: float):
    n = len(ranks_sorted)
    groups = []
    for i in range(n):
        j = i
        while j + 1 < n and (ranks_sorted[j + 1] - ranks_sorted[i]) < cd:
            j += 1
        if j > i:
            groups.append((i, j))
    # drop any group fully contained in another
    out = [(a, b) for (a, b) in groups
           if not any(a >= x and b <= y and (a, b) != (x, y) for (x, y) in groups)]
    return sorted(set(out))


# ─────────────────────────────────────────────────────────────────────────────
#  Draw the CD diagram  (Demsar layout: axis on top, methods elbowed to sides)
# ─────────────────────────────────────────────────────────────────────────────
def plot_cd(names, avranks, cd, n_datasets, fried_p, variant, out_stem):
    order   = np.argsort(avranks)
    names   = [names[i] for i in order]
    avranks = np.asarray(avranks, dtype=float)[order]
    k       = len(names)

    lowv  = int(math.floor(avranks.min()))
    highv = int(math.ceil(avranks.max()))
    if highv <= lowv:
        highv = lowv + 1

    cliques = find_cliques(avranks, cd)
    n_clq   = len(cliques)

    # ── geometry (data coords: x = rank, y = layout) ─────────────────────────
    axis_y    = 0.0
    line_gap  = 1.0
    clq_gap   = 0.45
    half      = (k + 1) // 2

    cd_bar_y  = axis_y + 1.4
    top       = cd_bar_y + 0.6
    label_top = axis_y - 0.6 - n_clq * clq_gap          # first label row
    bottom    = label_top - (half - 1) * line_gap - 0.8

    fig_h = 0.42 * half + 0.25 * n_clq + 2.2
    fig, ax = plt.subplots(figsize=(9.5, max(4.5, fig_h)))
    ax.set_xlim(lowv - 0.5, highv + 0.5)
    ax.set_ylim(bottom, top)
    ax.axis("off")

    # ── main rank axis + integer ticks ───────────────────────────────────────
    ax.plot([lowv, highv], [axis_y, axis_y], color="black", lw=1.3, zorder=3)
    for v in range(lowv, highv + 1):
        ax.plot([v, v], [axis_y, axis_y + 0.12], color="black", lw=1.0, zorder=3)
        ax.text(v, axis_y + 0.20, str(v), ha="center", va="bottom", fontsize=8)
        if v < highv:  # minor half-ticks
            ax.plot([v + 0.5, v + 0.5], [axis_y, axis_y + 0.06],
                    color="black", lw=0.8, zorder=3)

    # ── CD reference bar ─────────────────────────────────────────────────────
    ax.plot([lowv, lowv + cd], [cd_bar_y, cd_bar_y], color="black", lw=1.6)
    for xe in (lowv, lowv + cd):
        ax.plot([xe, xe], [cd_bar_y - 0.07, cd_bar_y + 0.07], color="black", lw=1.6)
    ax.text(lowv + cd / 2.0, cd_bar_y + 0.12, f"CD = {cd:.2f}",
            ha="center", va="bottom", fontsize=8.5, fontweight="bold")

    # ── connector lines + method labels ──────────────────────────────────────
    def connector(rank, label, y, side):
        x_end = (lowv - 0.5) if side == "left" else (highv + 0.5)
        ax.plot([rank, rank, x_end], [axis_y, y, y],
                color="#444444", lw=1.0, zorder=2)
        if side == "left":
            ax.text(x_end - 0.05, y, label, ha="right", va="center", fontsize=8.5)
        else:
            ax.text(x_end + 0.05, y, label, ha="left", va="center", fontsize=8.5)

    for i in range(half):                                   # best half -> left
        y = label_top - i * line_gap
        connector(avranks[i], f"{names[i]}  ({avranks[i]:.2f})", y, "left")
    for j, idx in enumerate(range(half, k)):                # rest -> right
        y = label_top - j * line_gap
        connector(avranks[idx], f"({avranks[idx]:.2f})  {names[idx]}", y, "right")

    # ── significance (clique) bars just below the axis ───────────────────────
    for level, (a, b) in enumerate(cliques):
        y = axis_y - 0.28 - level * clq_gap
        ax.plot([avranks[a] - 0.04, avranks[b] + 0.04], [y, y],
                color="black", lw=4.5, solid_capstyle="round", zorder=4)

    # ── titles ───────────────────────────────────────────────────────────────
    p_txt = f"{fried_p:.2e}" if fried_p < 1e-3 else f"{fried_p:.3f}"
    ax.set_title(
        f"Critical Difference Diagram — Parameter-Based Transfer "
        f"({variant.capitalize()})\n"
        f"Friedman p = {p_txt}   |   Nemenyi $\\alpha$ = {ALPHA}   |   "
        f"{k} methods, N = {n_datasets} datasets (settings $\\times$ backbones)",
        fontsize=10, fontweight="bold", pad=14,
    )
    ax.text((lowv + highv) / 2.0, bottom + 0.15,
            "lower average rank  =  better   •   methods joined by a bar are "
            "not significantly different",
            ha="center", va="bottom", fontsize=8, style="italic", color="#555555")

    fig.tight_layout()
    for ext in ("pdf", "png"):
        out = OUT_DIR / f"{out_stem}.{ext}"
        fig.savefig(out, dpi=600, bbox_inches="tight")
        print(f"    saved: {out}")
    plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
#  Run for both variants
# ─────────────────────────────────────────────────────────────────────────────
def run(variant: str, table_path: str):
    print("=" * 78)
    print(f"  {variant.upper()}")
    print("=" * 78)

    mae, names, dropped, n = load_matrix(table_path)
    k = len(names)
    if dropped:
        print(f"  Excluded {len(dropped)} method(s) without full coverage "
              f"(required by CD): {', '.join(dropped)}")

    avranks = mae.rank(axis=1, method="average", ascending=True).mean(axis=0).values
    stat, p = friedmanchisquare(*[mae[c].values for c in names])
    cd = q_alpha(k, ALPHA) * math.sqrt(k * (k + 1) / (6.0 * n))

    print(f"  methods = {k}   datasets = {n}")
    print(f"  Friedman chi2 = {stat:.2f},  p = {p:.3e}")
    print(f"  Critical Difference (alpha={ALPHA}) = {cd:.3f}")
    rank_order = sorted(zip(names, avranks), key=lambda t: t[1])
    print("  ranking (best -> worst):")
    for r, (nm, av) in enumerate(rank_order, 1):
        print(f"    {r:2d}. {nm:32s} avg-rank = {av:.2f}")
    if p >= ALPHA:
        print("  NOTE: Friedman not significant — differences may be due to chance.")

    plot_cd(names, avranks, cd, n, p, variant, f"cd_diagram_{variant}")
    print()


for variant, table in TABLES.items():
    run(variant, table)

print("=" * 78)
print("DONE →", OUT_DIR)
print("=" * 78)
